In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
class NaiveBayesClassifier:
    def __init__(self):
        self.classes = None
        self.priors = {}
        self.likelihoods = {}
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        for cls in self.classes:
            X_cls = X[y == cls]
            self.priors[cls] = len(X_cls) / len(X)
            
            self.likelihoods[cls] = {
                feature: (np.mean(X_cls[:, feature]), np.std(X_cls[:, feature]) + 1e-6)
                for feature in range(X.shape[1])
            }
    
    def _gaussian_pdf(self, x, mean, std):
        return (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(-((x - mean)**2 / (2 * std**2)))
    
    def predict(self, X):
        predictions = []
        for sample in X:
            posteriors = {}
            for cls in self.classes:
                prior = np.log(self.priors[cls])
                likelihood = sum(
                    np.log(self._gaussian_pdf(sample[feature], *self.likelihoods[cls][feature]))
                    for feature in range(len(sample))
                )
                posteriors[cls] = prior + likelihood
                
            predictions.append(max(posteriors, key=posteriors.get))
        return np.array(predictions)

In [ ]:

df = pd.read_parquet("../../data/benchmark/testing/eth")

X = df.drop(columns=['label']).to_numpy()
y = df['label'].to_numpy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
model = GaussianNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
nb = NaiveBayesClassifier()
nb.fit(X_train, y_train)

In [ ]:
y_pred = nb.predict(X_test)

/var/folders/8k/8gwvwkzs4gb02y8rr95frc040000gn/T/ipykernel_9662/113957339.py:31: RuntimeWarning: divide by zero encountered in log
  np.log(self._gaussian_pdf(sample[feature], *self.likelihoods[cls][feature]))


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.4183100767210751
Classification Report:
               precision    recall  f1-score   support

       False       0.98      0.26      0.42     15253
        True       0.27      0.98      0.42      4168

    accuracy                           0.42     19421
   macro avg       0.62      0.62      0.42     19421
weighted avg       0.83      0.42      0.42     19421

